# Compress and Evaluate Video Generation Models

| Component | Details |
|-----------|---------|
| **Goal** | Showcase a standard workflow for optimizing and evaluating a video generation model |
| **Model** |[Wan-AI/Wan2.1-T2V-1.3B](https://huggingface.co/Wan-AI/Wan2.1-T2V-1.3B) |
| **Dataset** |  [nannullna/laion_subset](https://huggingface.co/datasets/nannullna/laion_subset) |
| **Device** | 1 x H100 (80GB) |
| **Optimization Algorithms** | compiler(torch_compile), kernel(flash_attn3, sage_attn) |
| **Evaluation Metrics** | `total time`, `latency`, `througput`, `co2_emissions`, and `energy_consumed` |

## Getting Started

To install the required dependencies, you can run the following command:


In [ ]:
%pip install pruna
%pip install ftfy imageio imageio-ffmpeg

For more information about how to install Pruna, please refer to the [Installation](https://docs.pruna.ai/en/stable/setup/install.html) page.

Then, we will set the device to the best available option to maximize the optimization process's benefits. However, in this case, we recommend using a GPU.

In [1]:
import torch

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

## 1. Load the Model

First, we must load the original model using the diffusers library to ensure it fits into memory. In this example, we will use a light model compatible with most of the consumer-grade GPUs, [Wan-AI/Wan2.1-T2V-1.3B](https://huggingface.co/Wan-AI/Wan2.1-T2V-1.3B).

Pruna works at least as well with larger models, like the model version of Wan 2.1 14B or HuyuanVideo. The choice to use a smaller model is simply because it’s a good starting point, so feel free to use any [text-to-video model available on Hugging Face](https://huggingface.co/models?pipeline_tag=text-to-video&sort=trending).

In [2]:
from diffusers import AutoencoderKLWan, WanPipeline

model_id = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"

vae = AutoencoderKLWan.from_pretrained(model_id, subfolder="vae", torch_dtype=torch.float32)

pipe = WanPipeline.from_pretrained(model_id, vae=vae, torch_dtype=torch.bfloat16).to(device)

/home/marius/miniconda3/envs/tutorials/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Multiple distributions found for package optimum. Picked distribution: optimum
Skipping import of cpp extensions due to incompatible torch version 2.10.0+cu128 for torchao version 0.15.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
Loading pipeline components...: 100%|██████████| 5/5 [00:02<00:00,  2.29it/s]


Once we have loaded the pipeline, we can run some inference and check the output. The standard prompt structure for a video is **Subject + Subject Action + Scene**, which can become more complex as we add descriptions and details like the lighting, point of view, or visual style to achieve specific and refined results.

Remember that you can improve the quality of the video by increasing the number of frames, the number of inference steps, and the guidance scale, but this will also increment the time and amount of resources required to generate the video.

In [3]:
from diffusers.utils import export_to_video

prompt = "A dog runs on the beach, realistic."
negative_prompt = "Bright tones, overexposed, static, blurred details, subtitles, style, works, paintings, images, static, overall gray, worst quality, low quality, JPEG compression residue, ugly, incomplete, extra fingers, poorly drawn hands, poorly drawn faces, deformed, disfigured, misshapen limbs, fused fingers, still picture, messy background, three legs, many people in the background, walking backwards"  # noqa: E501

with torch.no_grad():
    output = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        height=480,
        width=832,
        num_frames=33,
        guidance_scale=3.0,
        num_inference_steps=15,
        generator=torch.Generator(device=device).manual_seed(42),
    ).frames[0]

export_to_video(output, "base_video.mp4", fps=15)

100%|██████████| 15/15 [00:11<00:00,  1.29it/s]


'base_video.mp4'

As we can see, the model has generated a nice short video based on our prompt.

## 2. Define the SmashConfigs

Now that we have correctly loaded and tested our base model, let's continue by defining the `SmashConfig`s to customize the optimizations we want to apply when smashing.

Take into account that not all optimization algorithms are available for all models, so you can learn about the requirements and compatibility in the [Algorithms Overview](https://docs.pruna.ai/en/stable/compression.html).

We will compare two attention kernels:
- [flash_attn3](https://docs.pruna.ai/en/stable/compression.html#flash-attn3) — Flash Attention 3 uses tiling, streaming and fusing for fast attention.
- [sage_attn](https://docs.pruna.ai/en/stable/compression.html#sage-attn) — SageAttention combines flash attention with quantization and smoothing.

Both are combined with [torch_compile](https://docs.pruna.ai/en/stable/compression.html#torch-compile) for maximum performance. Let's define a separate `SmashConfig` for each variant.

In [4]:
from pruna import SmashConfig

smash_config_fa3 = SmashConfig({
    "torch_compile": {"target": "module_list"},
    "flash_attn3": {},
})

smash_config_sage = SmashConfig({
    "torch_compile": {"target": "module_list"},
    "sage_attn": {},
})

INFO - Using best available device: 'cuda'
INFO - Using best available device: 'cuda'


## 3. Smash the Models

Next, we apply each `SmashConfig` to a separate copy of the pipeline. We keep a base copy for evaluation, and offload models to CPU between smash calls to stay within GPU memory.

Time to smash! Each call takes around 20 seconds, depending on the configuration.

In [5]:
import copy

from pruna import smash
from pruna.engine.utils import move_to_device

# Keep a base copy and a copy for sage_attn (both on CPU to save GPU memory)
base_pipe = copy.deepcopy(pipe).to("cpu")
sage_pipe = copy.deepcopy(pipe).to("cpu")

# Smash with flash_attn3 (pipe is already on GPU)
fa3_pipe = smash(model=pipe, smash_config=smash_config_fa3)

# Offload FA3 pipe to CPU, bring sage pipe to GPU, smash with sage_attn
move_to_device(fa3_pipe, "cpu")
move_to_device(sage_pipe, device)
sage_pipe = smash(model=sage_pipe, smash_config=smash_config_sage)

Fetching 7 files: 100%|██████████| 7/7 [00:00<00:00, 92911.80it/s]
INFO - Determined algorithm order: flash_attn3, torch_compile
INFO - Starting flash_attn3...
Fetching 7 files: 100%|██████████| 7/7 [00:00<00:00, 78293.67it/s]
Attention backends are an experimental feature and the API may be subject to change.
INFO - flash_attn3 was applied successfully.
INFO - Starting torch_compile...
INFO - torch_compile was applied successfully.
INFO - Determined algorithm order: sage_attn, torch_compile
INFO - Starting sage_attn...
Fetching 11 files: 100%|██████████| 11/11 [00:00<00:00, 151767.58it/s]
INFO - sage_attn was applied successfully.
INFO - Starting torch_compile...
INFO - torch_compile was applied successfully.


Now, we have two optimized models. Let's check how each works using the previous prompt. We run them sequentially, offloading to CPU between runs to keep GPU memory in check.

Consider that if you are using `torch_compile` as a compiler, you can expect the first inference warmup to take longer than the actual inference.

### Flash Attention 3


In [6]:
move_to_device(fa3_pipe, device)

with torch.no_grad():
    output_fa3 = fa3_pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        height=480,
        width=832,
        num_frames=33,
        guidance_scale=3.0,
        num_inference_steps=15,
        generator=torch.Generator(device=device).manual_seed(42),
    ).frames[0]

move_to_device(fa3_pipe, "cpu")
export_to_video(output_fa3, "fa3_video.mp4", fps=15)

  0%|          | 0/15 [00:00<?, ?it/s]/home/marius/miniconda3/envs/tutorials/lib/python3.11/site-packages/torch/_dynamo/variables/functions.py:2056: UserWarning: Dynamo detected a call to a `functools.lru_cache`-wrapped function. Dynamo ignores the cache wrapper and directly traces the wrapped function. Silent incorrectness is only a *potential* risk, not something we have observed. Enable TORCH_LOGS="+dynamo" for a DEBUG stack trace.
  torch._dynamo.utils.warn_once(msg)
100%|██████████| 15/15 [00:07<00:00,  1.98it/s]


'fa3_video.mp4'

### SageAttention

In [7]:
move_to_device(sage_pipe, device)

with torch.no_grad():
    output_sage = sage_pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        height=480,
        width=832,
        num_frames=33,
        guidance_scale=3.0,
        num_inference_steps=15,
        generator=torch.Generator(device=device).manual_seed(42),
    ).frames[0]

move_to_device(sage_pipe, "cpu")
export_to_video(output_sage, "sage_video.mp4", fps=15)


  0%|          | 0/15 [00:00<?, ?it/s]

100%|██████████| 15/15 [00:14<00:00,  1.05it/s]


'sage_video.mp4'

<video controls width="100%" style="max-width:640px; height:auto;">
    <source src="../assets/videos/video_generation/sage_video.mp4" type="video/mp4">
    Your browser does not support HTML5 video.
</video>

Both optimized models generate similar videos to the base model. If you notice a significant difference, it might be due to the model, the configuration, or the hardware. Feel free to reach out to us on [Discord](https://discord.gg/JFQmtFKCjd) if you have any questions.


## 4. Evaluate the Smashed Models

Now that we have our two smashed models, the key question is how much each has improved over the baseline. For this, we can run an evaluation of the performance using the `EvaluationAgent`. In this case, we will include metrics like `total_time`, `latency`, `throughput`, `co2_emissions`, and `energy_consumed`.

A complete list of the available metrics can be found in [Evaluation](https://docs.pruna.ai/en/stable/reference/evaluation.html).

In [8]:
from pruna import PrunaModel
from pruna.data.pruna_datamodule import PrunaDataModule
from pruna.evaluation.evaluation_agent import EvaluationAgent
from pruna.evaluation.metrics import (
    CO2EmissionsMetric,
    EnergyConsumedMetric,
    LatencyMetric,
    ThroughputMetric,
    TotalTimeMetric,
)
from pruna.evaluation.task import Task

# Define the metrics. Increment the number of iterations and
# warmup iterations to get a more accurate result.
metrics = [
    TotalTimeMetric(n_iterations=3, n_warmup_iterations=1),
    LatencyMetric(n_iterations=3, n_warmup_iterations=1),
    ThroughputMetric(n_iterations=3, n_warmup_iterations=1),
    CO2EmissionsMetric(n_iterations=3, n_warmup_iterations=1),
    EnergyConsumedMetric(n_iterations=3, n_warmup_iterations=1),
]

# Define the datamodule
datamodule = PrunaDataModule.from_string("LAION256")
datamodule.limit_datasets(10)

# Define the task and the evaluation agent
task = Task(metrics, datamodule=datamodule, device=device)
eval_agent = EvaluationAgent(task)

/home/marius/miniconda3/envs/tutorials/lib/python3.11/site-packages/thop/profile.py:12: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  if LooseVersion(torch.__version__) < LooseVersion("1.0.0"):
/home/marius/miniconda3/envs/tutorials/lib/python3.11/site-packages/thop/profile.py:68: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  if LooseVersion(torch.__version__) >= LooseVersion("1.1.0"):
INFO - Using best available device: 'cuda'
INFO - Using best available device: 'cuda'
INFO - Using best available device: 'cuda'
INFO - Using best available device: 'cuda'
INFO - Using best available device: 'cuda'
Generating object split: 100%|██████████| 114/114 [00:00<00:00, 678.42 examples/s]
INFO - Loaded only training, splitting train 80/10/10 into train, validation and test...
INFO - Testing compatibility with image_generation_collate...
INFO - Using provided list of metric instances.


In [9]:
# Evaluate the Flash Attention 3 model and offload it to CPU
move_to_device(fa3_pipe, device)
fa3_results = eval_agent.evaluate(fa3_pipe)
move_to_device(fa3_pipe, "cpu")

INFO - Using best available device: 'cuda'
INFO - Evaluating a smashed model.
INFO - Detected diffusers model. Using DiffuserHandler with fixed seed.
- The first element of the batch is passed as input.
- The generated outputs are expected to have .images attribute.
INFO - Evaluating stateful metrics.
INFO - Evaluating isolated inference metrics.
Measuring inference time:   0%|          | 0/3 [01:19<?, ?iter/s]


AttributeError: 'WanPipelineOutput' object has no attribute 'images'

In [ ]:
# Evaluate the SageAttention model and offload it to CPU
move_to_device(sage_pipe, device)
sage_results = eval_agent.evaluate(sage_pipe)
move_to_device(sage_pipe, "cpu")

In [ ]:
# Evaluate the base model and offload it to CPU
base_pipe = PrunaModel(model=base_pipe)
move_to_device(base_pipe, device)
base_results = eval_agent.evaluate(base_pipe)
move_to_device(base_pipe, "cpu")


Let's visualize and compare the evaluation results of the base model against both optimized variants.

In [ ]:
from IPython.display import Markdown, display  # noqa


def _improvement(base_metric, smashed_metric):  # noqa
    base_val = base_metric.result
    smashed_val = smashed_metric.result
    if base_metric.higher_is_better:
        return ((smashed_val - base_val) / base_val) * 100
    return ((base_val - smashed_val) / base_val) * 100


def make_comparison_table(base_results, fa3_results, sage_results):  # noqa
    header = "| Metric | Base | FA3 | FA3 Δ% | SageAttn | SageAttn Δ% |\n"
    header += "|" + "-----|" * 6 + "\n"
    rows = []
    for base, fa3, sage in zip(base_results, fa3_results, sage_results):
        unit = base.metric_units or ""
        fa3_diff = _improvement(base, fa3)
        sage_diff = _improvement(base, sage)
        row = (
            f"| {base.name} "
            f"| {base.result:.5f} {unit} "
            f"| {fa3.result:.5f} {unit} | {fa3_diff:+.2f}% "
            f"| {sage.result:.5f} {unit} | {sage_diff:+.2f}% |"
        )
        rows.append(row)
    return header + "\n".join(rows)


display(Markdown(make_comparison_table(base_results, fa3_results, sage_results)))

| Metric | Base Model | Smashed Model | Improvement % |
|-----|-----|-----|-----|
| total_time | 460992.1875000 ms | 265793.1718750 ms | 42.34% |
| latency | 153664.0625000 ms/num_iterations | 88597.7239583 ms/num_iterations | 42.34% |
| throughput | 0.0000065 num_iterations/ms | 0.0000113 num_iterations/ms | 73.44% |
| co2_emissions | 0.0031181 kgCO2e | 0.0018072 kgCO2e | 42.04% |
| energy_consumed | 0.0556424 kWh | 0.0322483 kWh | 42.04% |

Both optimized models are significantly faster than the base model. The table above lets you compare Flash Attention 3 and SageAttention side-by-side in terms of latency, throughput, energy consumption, and CO₂ emissions. Both kernels, combined with `torch_compile`, deliver substantial speed-ups while maintaining output quality.

We can save either optimized model to disk or share it on the Hub. Note that `torch_compile` is device-dependent and will be re-applied when loading the model on a different device.

In [ ]:
# Save the Flash Attention 3 model to disk
fa3_pipe.save_pretrained("Wan2.1-T2V-1.3B-fa3")
# Load from disk:  PrunaModel.from_pretrained("Wan2.1-T2V-1.3B-fa3/")

# Save the SageAttention model to disk
sage_pipe.save_pretrained("Wan2.1-T2V-1.3B-sage")
# Load from disk:  PrunaModel.from_pretrained("Wan2.1-T2V-1.3B-sage/")

## Conclusions

In this tutorial, we compared two attention kernel optimizations — **Flash Attention 3** and **SageAttention** — for a text-to-video model.

We loaded the base model, defined a separate `SmashConfig` for each kernel (both combined with `torch_compile`), and smashed two copies of the pipeline. We then evaluated all three variants (base, FA3, SageAttn) using the `EvaluationAgent`.

The results show that both kernels significantly increase inference speed and reduce energy consumption while maintaining output quality. The side-by-side comparison makes it easy to pick the best optimization strategy for your specific use case.

Check out our other [tutorials](https://docs.pruna.ai/en/stable/docs_pruna/tutorials/index.html) for more examples on how to optimize and evaluate image generation models or LLM models.